In [1]:
library(SPOTlight)
library(Seurat)
library(ggplot2)
library(SingleCellExperiment)
library(scater)
library(scran)
library(dplyr)

The legacy packages maptools, rgdal, and rgeos, underpinning this package
will retire shortly. Please refer to R-spatial evolution reports on
https://r-spatial.org/r/2023/05/15/evolution4.html for details.
This package is now running under evolution status 0 

Attaching SeuratObject

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘matrixStats’


The following objects are masked from ‘package:Biobase’:

    anyMissing, rowMedians



Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colV

In [2]:
T_names = c( "T993")
for (profile in T_names){
    sc_data = readRDS("/data/snRNA_downsample.rds") #######
    st_data = readRDS(paste0("/data/",profile,"cellbin.rds"))
    sce <- as.SingleCellExperiment(sc_data)
    spe = as.SingleCellExperiment(st_data)
    ### Feature selection
    sce <- logNormCounts(sce)

    ### Variance modelling
    # Remove ribosomal and mitochondrial genes.
    genes <- !grepl(pattern = "^RP[L|S]|MT", x = rownames(sce))
    dec <- modelGeneVar(sce , subset.row = genes)

    # Calculate hypervariable genes.
    hvg <- getTopHVGs(dec, n = 1000)

    # Add cell annotation information.
    colLabels(sce) <- colData(sce)$subclass.v4

    # Compute marker genes
    mgs <- scoreMarkers(sce, subset.row = genes)

    mgs_fil <- lapply(names(mgs), function(i) {
        x <- mgs[[i]]
        # Filter and keep relevant marker genes, those with AUC > 0.8
        x <- x[x$mean.AUC > 0.5, ]
        # Sort the genes from highest to lowest weight
        x <- x[order(x$mean.AUC, decreasing = TRUE), ]
        # Add gene and cluster id to the dataframe
        x$gene <- rownames(x)
        x$cluster <- i
        data.frame(x)
    })
    mgs_df <- do.call(rbind, mgs_fil)

    idx <- split(seq(ncol(sce)), sce$subclass.v4)
    # downsample to at most 20 per identity & subset
    # We are using 5 here to speed up the process but set to 75-100 for your real
    # life analysis
    n_cells <- 50
    cs_keep <- lapply(idx, function(i) {
        n <- length(i)
        if (n < n_cells)
            n_cells <- n
        sample(i, n_cells)
    })
    sce <- sce[, unlist(cs_keep)]

    res <- SPOTlight(
        x = sce,
        y = spe,
        groups = as.character(sce$subclass.v4),
        mgs = mgs_df,
        hvg = hvg,
        weight_id = "mean.AUC",
        group_id = "cluster",
        gene_id = "gene")

    #Extract data from `SPOTlight`:
    decon_mtrx <- res$mat
    #rename
    colnames(decon_mtrx) <- paste(colnames(decon_mtrx),"Spotlight",sep = "_")
    write.csv(decon_mtrx,paste0('/data/work/Layer division/01_Result/Spotlight_',profile,'.cellbin_prct.csv'))
}

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 13.2 GiB”
Scaling count matrix

Seeding initial matrices

Training NMF model

Deconvoluting mixture data

